First you need to get your API key. So need to log in in fireping. the information for the code i got from: https://fireping.net/api/docs. You need to get 200 for the response.status_code: means Response containing a list of locations

In [6]:
import requests
import pandas as pd

# My API key
api_key = "fp_b1240f0a75d60c45b30f200372d5b9206b866b053de347d168dc9e00abd438da" #need to put your own!

# API endpoint
url = "https://fireping.net/api/v1/locations"  

# Headers for authentication
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# Make the GET request
response = requests.get(url, headers=headers)

print(response.status_code)


200


Next define the location. Choose the middle from where you want the radius. Can also go look in the offical fireping site for places where there is fireactivity and then choose a coordinate
Need the respone 201: means Response containing a single location

In [7]:
payload = {
    "latitude": 19.0377,
    "longitude": -90.6825,
    "name": "Mexico Test", #Give a name to the data you want to download
    "radius": 25000   #maximum radius without paying
}

response = requests.post(url, headers=headers, json=payload)

print(response.status_code)
print(response.json())

201
{'data': {'id': '8d4ae3b4-beca-48d0-8e60-f8b429c51161', 'name': 'Mexico Test', 'latitude': 19.0377, 'longitude': -90.6825, 'radius': 25000, 'enabled': True}}


Now we get the fire data of the location we chose. SO we have defined a location named mexico test and we can now get the data. Save the data as fires.

In [8]:
# Append query string directly in URL
url = "https://fireping.net/api/v1/fires/user?hours=168&limit=1000" #the maximum amount we are allowed to get is 24 houts and 1000 points

response = requests.get(url, headers=headers)

print(response.status_code)
fires = response.json()

200


Can also save the retrieved fire data of today in a csv file for reproducibility

In [ ]:
df = pd.DataFrame(fires['data'])
df.to_csv(f"../data/processed/fire_data_insert_your_date&place.csv", index=False)
print("Data saved successfully.")

Now we create the interactive map using folium

In [11]:
import folium
from folium.plugins import MarkerCluster
from branca.colormap import linear

df = pd.DataFrame(fires['data'])
#df = pd.read_csv("../data/processed/fire_data_insert_your_date&place.csv"), can also use the csv

# Base Map
Custom_tile_url = 'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png' 
Custom_attribution = (
'&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors '
'&copy; <a href="https://carto.com/attributions">CARTO</a>'
)

m = folium.Map(
    location=[19.038, -90.683],
    zoom_start=10,
    tiles=Custom_tile_url,
    attr=Custom_attribution,
)

#Shows the extent of data aquisition area
radius_m = 25000
folium.Circle(
    location=[19.0377, -90.6825],
    radius=radius_m,       # in meters
    color="blue",
    fill=True,
    fill_opacity=0.1,
).add_to(m)

#Create a cluster layer and a raw data points layer so can switch between them
cluster_layer = folium.FeatureGroup(
    name="Clustered Fires (Whole data)"
).add_to(m)
raw_layer = folium.FeatureGroup(
    name="Raw Fire Points (Whole data)"
).add_to(m)

#Place the cluster inside feature group so Layercontrol can turn on and off
marker_cluster = MarkerCluster().add_to(cluster_layer)

# Change the shortcuts to full word for better comprehension
confidence_map = {"l": "Low", "n": "Nominal", "h": "High"}

#Colors for the FRP Sign
min_frp = df["frp"].min()
max_frp = df["frp"].max()
colormap = linear.YlOrRd_09.scale(min_frp, max_frp)

#Convert timestamps to datetime detected ->(exp: 2026-05-16T07:32:00Z)
df['detected_at'] = pd.to_datetime(df['detected_at'])
#Create new column that shows the data without time
df['date_only'] = df['detected_at'].dt.date #(exp: 2026-05-16)
#finds all different dates and puts them in chronological order
unique_dates = sorted(df['date_only'].unique()) 
#Create for each day in week a layer
daily_layers = {}
for date in unique_dates:
    date_layer = folium.FeatureGroup(
        name=str(date)
    ).add_to(m)
    daily_layers[date] = date_layer


for _, fire in df.iterrows():
    # Split date and time
    timestamp = fire['detected_at']
    formatted_date = timestamp.strftime("%d.%m.%Y")
    time = timestamp.strftime("%H:%M")
    
    # Convert confidence
    conf_text = confidence_map.get(fire['confidence'], fire['confidence'])

    #determine which point belong to which day
    fire_date = fire['date_only']

    # Popup text
    popup_text = (
        f"Date: {formatted_date}<br>"
        f"Time: {time}<br>"
        f"Confidence: {conf_text}<br>"
        f"FRP: {fire['frp']} MW<br>"
        f"Satellite: {fire['satellite']}"
    )
    
    folium.CircleMarker(
        location=[fire['latitude'], fire['longitude']],
        radius=6,
        popup=folium.Popup(popup_text, max_width=250),
        color=colormap(fire['frp']),
        fill_color=colormap(fire['frp']),
        fill=True,
        fill_opacity=0.7
    ).add_to(marker_cluster)

    folium.CircleMarker(
        location=[fire['latitude'], fire['longitude']],
        radius=6,
        popup=folium.Popup(popup_text, max_width=250),
        color=colormap(fire['frp']),
        fill=True,
        fill_color=colormap(fire['frp']),
        fill_opacity=0.8
    ).add_to(raw_layer)

    folium.CircleMarker(
        location=[fire['latitude'], fire['longitude']],
        radius=6,
        popup=folium.Popup(popup_text, max_width=250),
        color=colormap(fire['frp']),
        fill=True,
        fill_color=colormap(fire['frp']),
        fill_opacity=0.8
    ).add_to(daily_layers[fire_date])


# Legend for FRP Signs
colormap.caption = 'Fire Radiative Power (MW)' 
colormap.add_to(m)
m.get_root().html.add_child(folium.Element("""
<style>
.legend {
    background-color: rgba(255,255,255,0.8) !important; 
    padding: 10px;
    border-radius: 5px;
}
</style>
""")) #change background colour of Label to white so can see on black basemap
# White background, opacity 0.8
# padding adds spacing, so it doesn't touch edges
# 5px makes smooth rounded edges
# injecting custom HTML/CSS styling

#for controlling the layers clustered or raw
folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element("""
<style>
.leaflet-control-layers {
    width: 130px !important;
    font-size: 11px;
}
</style>
"""))

#Save the map as html in outputs
display(m)
m.save("../outputs/wildfire_map.html")